### 1. BÀI 1: BIỂU DIỄN DỮ LIỆU THÀNH MA TRẬN & ĐỘ TƯƠNG ĐỒNG

---

#### 1.1. Biến dữ liệu thành ma trận

In [ ]:
import numpy as np
from PIL import Image
import os


image_files = [f"anh{i}.jpg" for i in range(1, 9)] 
X_list = []

# Kích thước chuẩn
TARGET_SIZE = (100, 100) 

for file in image_files:
    if os.path.exists(file):
        # Đọc ảnh, chuyển sang xám (L), resize cho đồng nhất và ép kiểu float
        img = Image.open(file).convert("L").resize(TARGET_SIZE)
        img_array = np.array(img, dtype=float)
        
        # Duỗi ảnh thành vector 1D
        X_list.append(img_array.flatten())
    else:
        print(f"Cảnh báo: Không tìm thấy {file}")
        # Khởi tạo một mảng toàn số 0 để code không bị lỗi nếu thiếu ảnh (chỉ dùng để test)
        X_list.append(np.zeros(TARGET_SIZE[0] * TARGET_SIZE[1]))

# Tạo ma trận X từ danh sách 8 vector ảnh
X = np.array(X_list)

print("Kích thước X.shape:", X.shape)
print("Giải thích:")
print(f"- Số hàng: {X.shape[0]} (đại diện cho 8 bức ảnh)")
print(f"- Số cột: {X.shape[1]} (đại diện cho số lượng pixel của mỗi ảnh, là {TARGET_SIZE[0]} x {TARGET_SIZE[1]})")

#### 1.2. Phép toán cơ bản

In [ ]:
# Tính vector trung bình theo cột (axis=0)
mean_vector = np.mean(X, axis=0)

print("Shape của X trước khi trừ:", X.shape)         # (8, H*W)
print("Shape của mean_vector:", mean_vector.shape)         # (H*W,)

# Trừ trung bình: Numpy sẽ copy mean_vector thành 8 bản sao để trừ cho từng hàng của X
X_centered = X - mean_vector

print("Shape của X sau khi trừ trung bình:", X_centered.shape) # (8, H*W)

#### 1.3. Cosine similarity

In [ ]:
def cosine_similarity(X, Y=None):
    if Y is None:
        Y = X
    Xn = X / np.linalg.norm(X, axis=1, keepdims=True) 
    Yn = Y / np.linalg.norm(Y, axis=1, keepdims=True) 
    return Xn @ Yn.T  

#### 1.4. Truy vấn

In [ ]:
def search(query, dataset, top_k=3):
    if query.ndim == 1:
        query = query.reshape(1, -1)
        
    sim_scores = cosine_similarity(query, dataset)
    sim_scores_1d = sim_scores[0]
    
    # Lấy ra index của top_k ảnh có điểm cosine cao nhất
    top_indices = np.argsort(sim_scores_1d)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append((idx, sim_scores_1d[idx]))
    return results

print("\n--- Kết quả truy vấn ---")
# Lấy ảnh ở vị trí index 0 (anh_1.jpg) làm query
query_image = X[0:1] 

# Tìm top 3 ảnh giống nhất trong tập X
ket_qua = search(query_image, X, top_k=3)

print("Top 3 ảnh giống 'anh_1.jpg' nhất (bao gồm chính nó):")
for i, (idx, score) in enumerate(ket_qua):
    print(f"Hạng {i+1}: Ảnh số {idx + 1} (anh_{idx + 1}.jpg) - Điểm tương đồng: {score:.4f}")

#### 1.5. Nhận xét

In [ ]:
print("\n--- Ma trận tương đồng của tất cả các cặp ---")
# Tính độ tương đồng của mọi cặp trong 8 ảnh
sim_matrix = cosine_similarity(X)


# Tìm cặp giống nhất và khác nhất (bỏ qua đường chéo chính - ảnh so với chính nó)
np.fill_diagonal(sim_matrix, -2) # Gán đường chéo bằng -2 để loại trừ khi tìm max
max_idx = np.unravel_index(np.argmax(sim_matrix), sim_matrix.shape)

np.fill_diagonal(sim_matrix, 2)  # Gán đường chéo bằng 2 để loại trừ khi tìm min
min_idx = np.unravel_index(np.argmin(sim_matrix), sim_matrix.shape)

print(f"Cặp giống nhau nhất: Ảnh {max_idx[0]+1} và Ảnh {max_idx[1]+1}")
print(f"Cặp khác biệt nhất: Ảnh {min_idx[0]+1} và Ảnh {min_idx[1]+1}")

#### 1.6. Nhận xét trực giác
* Ảnh 4 và ảnh 5 có cùng tôn màu sáng, hình dáng của vật thể giống nhau nên nó là cặp ảnh giống nhau nhất
* Ảnh 2 và ảnh 8 khác tông màu, background, hình dáng của vật thể nên nó là cặp ảnh khác biệt nhau nhất
- Vậy theo trực giác kết quả đúng với dự đoán

---

### 2. BÀI 2: BIẾN ĐỔI TUYẾN TÍNH & SVD

#### 2.1. Biến đổi tuyến tính 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Tạo một hình vuông đơn giản với các điểm A(0,0), B(1,0), C(1,1), D(0,1)
points = np.array([[0, 1, 1, 0, 0],
                   [0, 0, 1, 1, 0]])

# Ma trận quay R
theta = np.radians(45)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

# Ma trận co giãn S (x scale 2 lần, y không đổi)
S = np.array([[2, 0],
              [0, 1]])

# Áp dụng phép biến đổi: Nhân ma trận
points_rotated = R @ points
points_scaled = S @ points
points_combined = (R @ S) @ points # Vừa scale vừa quay

# Vẽ ảnh trước và sau bằng Matplotlib
plt.figure(figsize=(12, 4))

plt.subplot(1, 4, 1)
plt.plot(points[0], points[1], marker='o'); plt.title("Gốc")
plt.xlim(-2, 3); plt.ylim(-1, 3); plt.grid(True)

plt.subplot(1, 4, 2)
plt.plot(points_rotated[0], points_rotated[1], marker='o', color='r'); plt.title("Quay 45°")
plt.xlim(-2, 3); plt.ylim(-1, 3); plt.grid(True)

plt.subplot(1, 4, 3)
plt.plot(points_scaled[0], points_scaled[1], marker='o', color='g'); plt.title("Dãn X(x2)")
plt.xlim(-2, 3); plt.ylim(-1, 3); plt.grid(True)

plt.subplot(1, 4, 4)
plt.plot(points_combined[0], points_combined[1], marker='o', color='purple'); plt.title("Quay + Dãn X")
plt.xlim(-2, 3); plt.ylim(-1, 3); plt.grid(True)

plt.tight_layout()
plt.show()

#### 2.2. Nén ảnh bằng SVD

In [ ]:
from PIL import Image

# Đọc ảnh và chuyển thành ma trận M 
M = np.array(Image.open("anh7.jpg").convert("L"), dtype=float)

# Phân rã SVD
U, S, Vt = np.linalg.svd(M, full_matrices=False)

# Hàm tái tạo ảnh với k giá trị kỳ dị 
def reconstruct(k):
    return (U[:, :k] * S[:k]) @ Vt[:k, :]

# Tái tạo và hiển thị ảnh để so sánh
ks = [5, 20, 50]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Ảnh gốc
axes[0].imshow(M, cmap='gray')
axes[0].set_title('Ảnh gốc')
axes[0].axis('off')

# Ảnh tái tạo
for i, k in enumerate(ks):
    M_k = reconstruct(k)
    axes[i+1].imshow(M_k, cmap='gray')
    axes[i+1].set_title(f'k = {k}')
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()

#### 2.3. Đánh giá 
* Dữ liệu ảnh gốc: Kích thước ảnh là H x W, số phần tử cần lưu là $H * W$. 
* Dữ liệu khi nén SVD với K: U có kích thước ($H * W$), $V^T$ có kích thước ($K * W$) và S gồm K giá trị. 
* Khi đó tổng số phần tử là $K * (H + W + 1)$.
* Tỉ lệ nén: $\text{Ratio} = \frac{\text{Kích thước nén}}{\text{Kích thước gốc}}$.  
* Sai số (MSE - Mean Squared Error): Trung bình bình phương khoảng cách giữa pixel ảnh gốc và ảnh tái tạo.

In [ ]:
H, W = M.shape
size_original = H * W

print(f"{'k':<5} | {'Tỉ lệ nén (%)':<15} | {'Sai số (MSE)':<15}")
print("-" * 40)

# Khai báo sẵn các giá trị k cần test
test_ks = [1, 5, 10, 15, 20, 30, 40, 50, 60, 70, 80, 90, 100]
errors = []

for k in test_ks:
    M_k = reconstruct(k)
    
    # Tính sai số (MSE)
    mse = np.mean((M - M_k)**2)
    errors.append(mse)
    
    # In kết quả cho các k cụ thể
    if k in [5, 20, 50]:
        size_compressed = k * (H + W + 1)
        compression_ratio = (size_compressed / size_original) * 100
        print(f"{k:<5} | {compression_ratio:<14.2f}% | {mse:<15.2f}")

# Vẽ đồ thị sai số theo k
plt.figure(figsize=(8, 4))
plt.plot(test_ks, errors, marker='o', linestyle='-', color='b')
plt.title("Đồ thị Sai số tái tạo (MSE) theo k")
plt.xlabel("Số lượng giá trị kỳ dị (k)")
plt.ylabel("Sai số tái tạo (MSE)")
plt.grid(True)
plt.show()

#### 2.4. Nhận xét
* Với ảnh có kích thước khoảng 500x500, k nằm trong khoảng 30 đến 50 là hình ảnh đã đủ rõ ràng để nhận diện bằng mắt thường,    trong khi dung lượng lưu trữ chỉ còn khoảng 10-20% so với gốc.
* Liên hệ với giảm chiều và nén dữ liệu trong AI: SVD là cốt lõi của PCA (Principal Component Analysis - Phân tích thành phần chính). Thay vì giữ lại toàn bộ đặc trưng (pixel) của dữ liệu, AI dùng SVD để tìm ra các "trục" (thành phần) quan trọng nhất chứa nhiều thông tin nhất (tương ứng với các giá trị kỳ dị lớn). Bằng cách vứt bỏ các giá trị kỳ dị nhỏ (nhiễu, chi tiết vụn vặt), AI vừa nén được dữ liệu (chạy nhanh hơn, tốn ít RAM hơn) vừa chống được hiện tượng quá khớp (overfitting).
